# Example 10.1.2 / 1: Continuous Flows with an Integral Optimum

Companion notebook for **10.1.2 Total Unimodularity and Integral Flows** in the AMPL lab Documentation, section "Continuous Flows with an Integral Optimum". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Continuous Flows with an Integral Optimum

A plant `s` sends 7 units to a customer `t` through two warehouses `a` and `b`, with an arc
from `a` to `b` as well. All supplies, demands and capacities are integers, and `Flow` is
declared continuous:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;                                   # plant, warehouses, customer
set ARCS within NODES cross NODES;           # directed connections
param balance {NODES};                       # net supply; negative means demand
param capacity {ARCS} >= 0;                  # largest flow on the arc
param unit_cost {ARCS} >= 0;                 # cost per unit of flow

var Flow {(i, j) in ARCS} >= 0, <= capacity[i, j];   # continuous

minimize TotalCost: sum {(i, j) in ARCS} unit_cost[i, j] * Flow[i, j];

subject to FlowBalance {k in NODES}:
    sum {(k, j) in ARCS} Flow[k, j] - sum {(i, k) in ARCS} Flow[i, k] = balance[k];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["s", "a", "b", "t"]
ampl.set["ARCS"] = [("s", "a"), ("s", "b"), ("a", "b"), ("a", "t"), ("b", "t")]
ampl.param["balance"] = {"s": 7, "a": 0, "b": 0, "t": -7}
ampl.param["capacity"] = {("s", "a"): 4, ("s", "b"): 5, ("a", "b"): 2,
                          ("a", "t"): 3, ("b", "t"): 6}
ampl.param["unit_cost"] = {("s", "a"): 1, ("s", "b"): 3, ("a", "b"): 1,
                           ("a", "t"): 4, ("b", "t"): 1}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalCost"].value())
flows = ampl.var["Flow"].to_dict()
print(flows)
print("all integral:", all(abs(v - round(v)) < 1e-9 for v in flows.values()))

solved 27.0
{('a', 'b'): 2, ('a', 't'): 1, ('b', 't'): 6, ('s', 'a'): 3, ('s', 'b'): 4}
all integral: True


### Pure AMPL: data.dat and model.run

The LP returned whole numbers on every arc although nothing declared `Flow` integer: the
optimum is an extreme point of a polyhedron whose extreme points are all integral.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NODES := s a b t;
set ARCS := (s,a) (s,b) (a,b) (a,t) (b,t);
param balance := s 7  a 0  b 0  t -7;
param:      capacity unit_cost :=
  s a          4         1
  s b          5         3
  a b          2         1
  a t          3         4
  b t          6         1 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display TotalCost, Flow;

TotalCost = 27

Flow :=
a b   2
a t   1
b t   6
s a   3
s b   4
;



### Another way in amplpy

Change the capacity of arc `a -> b` to the fractional value 2.5 and the guarantee no longer
applies, because one bound is no longer an integer:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["NODES"] = ["s", "a", "b", "t"]
ampl.set["ARCS"] = [("s", "a"), ("s", "b"), ("a", "b"), ("a", "t"), ("b", "t")]
ampl.param["balance"] = {"s": 7, "a": 0, "b": 0, "t": -7}
ampl.param["capacity"] = {("s", "a"): 4, ("s", "b"): 5, ("a", "b"): 2,
                          ("a", "t"): 3, ("b", "t"): 6}
ampl.param["unit_cost"] = {("s", "a"): 1, ("s", "b"): 3, ("a", "b"): 1,
                           ("a", "t"): 4, ("b", "t"): 1}

In [11]:
# run.py
ampl.param["capacity"][("a", "b")] = 2.5
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["TotalCost"].value())
print(ampl.var["Flow"].to_dict())

solved 26.5
{('a', 'b'): 2.5, ('a', 't'): 1, ('b', 't'): 6, ('s', 'a'): 3.5, ('s', 'b'): 3.5}
